In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

CSV_PATH = r"raw_loan_dataset.csv"
OUT_PATH = r"clean_loan_dataset.csv"

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/raw_loan_dataset.csv'

# B1 — Loan Data Processing

This notebook reproduces the Lesson 5 preprocessing pipeline on the raw loan dataset.
Exception: Step 10 uses **RobustScaler** instead of StandardScaler (see explanation below).

## Step 1 — Load & Inspect

**Key issues observed:**
- `Income` and `LoanAmount` contain `$` signs and commas, preventing direct numeric conversion.
- `HasCollateral`, `PreviousDefaults`, and `Approved` have inconsistent spellings (e.g., "yes", "y", "yse", "approved").
- Some rows have missing values across multiple columns.

In [ ]:
df = pd.read_csv(CSV_PATH)

print("=== INITIAL HEAD ===")
print(df.head())

print("\n=== INITIAL INFO ===")
print(df.info())

print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())

## Step 2 — Clean Currency Formatting

In [ ]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

print("Currency symbols removed. Sample values:")
print(df[["Income", "LoanAmount"]].head())

## Step 3 — Fix Category Errors Before Imputation

In [ ]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = (
    df["HasCollateral"]
    .astype(str).str.strip().str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

print("HasCollateral value counts after normalisation:")
print(df["HasCollateral"].value_counts(dropna=False))

df["PreviousDefaults"] = (
    df["PreviousDefaults"]
    .astype(str).str.strip().str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

print("\nPreviousDefaults value counts after normalisation:")
print(df["PreviousDefaults"].value_counts(dropna=False))

df["Approved"] = (
    df["Approved"]
    .astype(str).str.strip().str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

print("\nApproved value counts after normalisation:")
print(df["Approved"].value_counts(dropna=False))

## Step 4 — Impute Missing Values

In [ ]:
# Numeric columns: median imputation
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

# Categorical columns: mode imputation
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

print("=== MISSING VALUES AFTER IMPUTATION ===")
print(df.isnull().sum())

## Step 5 — Remove Duplicates

In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"Rows before duplicates removal: {before}")
print(f"Rows after duplicates removal: {df.shape[0]}")
print(f"Duplicates removed: {before - df.shape[0]}")

## Step 6 — Outliers (IQR Capping)

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_bounds(df["Income"])
low_credit,  high_credit  = iqr_bounds(df["CreditScore"])
low_loan,    high_loan    = iqr_bounds(df["LoanAmount"])
low_emp,     high_emp     = iqr_bounds(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

print("Outliers capped using IQR (k=1.5).")
print(f"Income bounds:     [{low_income:.2f}, {high_income:.2f}]")
print(f"CreditScore bounds: [{low_credit:.2f}, {high_credit:.2f}]")
print(f"LoanAmount bounds:  [{low_loan:.2f}, {high_loan:.2f}]")
print(f"EmploymentYears bounds: [{low_emp:.2f}, {high_emp:.2f}]")

## Step 7 — Label Encoding

In [ ]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("=== CLASS DISTRIBUTION (Approved after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)

## Step 8 — Class Balance Check

In [ ]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
print(f"Minority class proportion: {class_ratio:.3f}")

if class_ratio < 0.30:
    print("Warning: severely imbalanced classes — consider balancing techniques.")
else:
    print("Class balance OK for baseline Accuracy (both classes well represented).")

## Step 9 — Feature Engineering (No Leakage)

In [ ]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

print("Engineered features added: DebtToIncome, IncomePerYearEmployed")
print(df[["DebtToIncome", "IncomePerYearEmployed"]].describe().round(3))

## Step 10 — Feature Scaling (RobustScaler)

### Scaler Choice: RobustScaler

I chose **RobustScaler** from `sklearn.preprocessing` instead of StandardScaler.

**Why it fits this loan dataset:**
RobustScaler scales features using the median and the interquartile range (IQR) rather than the mean and standard deviation. This makes it robust to outliers — even though we applied IQR capping in Step 6, financial data like income and loan amounts often have skewed distributions where the mean can be pulled by remaining extreme values. The median and IQR are not affected by such skewness, making RobustScaler a more appropriate choice for financial datasets. Additionally, RobustScaler does not assume the data is normally distributed, which is a reasonable assumption for loan application features.

**Source used for research:**
scikit-learn documentation — "Compare the effect of different scalers on data with outliers":
https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html

In [ ]:
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

print(f"Columns to scale: {scale_cols}")
print(f"Binary columns excluded from scaling: {binary_cols}")

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("\nScaling complete. Sample of scaled numeric features:")
print(df[scale_cols].head())

## Step 11 — Final Checks & Save

In [ ]:
print("=== FINAL HEAD ===")
print(df.head())

print("\n=== FINAL INFO ===")
print(df.info())

print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())

df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")